# 105. Visual attention 을 이용한 Image captioning

- Google Tutorial https://www.tensorflow.org/tutorials/text/image_captioning?hl=ko 참조  

- MSCOCO dataset 대신 Flicker8k dataset 으로 변경

아래 사진에서 "a surfer riding on a wave" 가 caption 되도록 attention-based model 작성

![Man Surfing](https://tensorflow.org/images/surf.jpg)

*[Image Source](https://commons.wikimedia.org/wiki/Surfing#/media/File:Surfing_in_Hawaii.jpg); License: Public Domain*

- attention based model 은 image 의 어떤 부분을 caption 생성 도중 주목할 것인지 가능하도록 해 준다.

![Prediction](https://tensorflow.org/images/imcap_prediction.png)

- model architecture 는 [Show, Attend and Tell: Neural Image Caption Generation with Visual Attention](https://arxiv.org/abs/1502.03044) 와 유사. 

In [0]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    root_captioning = "/content/drive/My Drive/Flicker8"
    print("Note: using Google CoLab")
    %tensorflow_version 2.x
except:
    print("Note: not using Google CoLab")
    root_captioning = r'./Flicker8'
    
import tensorflow as tf
tf.__version__

In [0]:
import matplotlib.pyplot as plt

# Scikit-learn includes many helpful utilities
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

import re
import numpy as np
import os
import time
import json
from PIL import Image
import pickle
from tqdm import tqdm
import string

## Download and prepare the Flicker8k dataset

In [0]:
filename = root_captioning + "/Flickr8k_text/Flickr8k.token.txt"

fp = open(filename, 'r')

null_punct = str.maketrans('', '', string.punctuation)
lookup = dict()

max_length = 0

for line in fp.read().split('\n'):      
    tokens = line.split()
    if len(line) >= 2:
        id = tokens[0].split('.')[0]       #1000268201_693b08cb0e
        desc = tokens[1:]       #['A', 'girl', 'going', 'into', 'a', 'wooden', 'building', '.']
       # Cleanup description
        desc = [word.lower() for word in desc]
        desc = [w.translate(null_punct) for w in desc]
        desc = [word for word in desc if len(word)>1]    # a 같은 한글자 단어 제외
        desc = [word for word in desc if word.isalpha()]  # 알파벳 문자외는 제외
        max_length = max(max_length, len(desc))        # 가장 긴 description 길이 (단어수)

        if id not in lookup:
            lookup[id] = list()
        lookup[id].append(' '.join(desc))

In [0]:
annotations = list()

for id, captions in lookup.items():
    for caption in captions:
        annot = dict()
        annot['image_id'] = id
        annot['caption'] = caption
        annotations.append(annot)

annotations[:3]

In [0]:
img_directory = root_captioning +  '/Flickr8k_Dataset/Flicker8k_Dataset/'
imgs = os.listdir(img_directory)
print(len(imgs))
print(imgs[:5])

In [0]:
# 모든 caption 과 image name 을 하나의 list 에 저장
all_captions = []
all_img_name_vector = []

for annot in annotations:
    caption = '<start> ' + annot['caption'] + ' <end>'
    image_id = annot['image_id']
    full_coco_image_path = img_directory +  image_id + '.jpg'
    if os.path.exists(full_coco_image_path):
        all_img_name_vector.append(full_coco_image_path)
        all_captions.append(caption)
    else:
        print(full_coco_image_path)

In [0]:
train_captions, img_name_vector = shuffle(all_captions,
                                    all_img_name_vector, random_state=1)
len(train_captions), len(img_name_vector)

In [0]:
num_examples = 6000

train_captions = train_captions[:num_examples]
img_name_vector = img_name_vector[:num_examples]

In [0]:
len(train_captions), len(img_name_vector)

In [0]:
# sanity check
exist_cnt = 0
not_exist_cnt = 0

for path in img_name_vector:
    if os.path.exists(path):
        exist_cnt += 1
    else:
        not_exist_cnt += 1
        print(path)

exist_cnt, not_exist_cnt

## InceptionV3 를 이용한 image preprocess
- pre-trained InceptionV3 를 이용하여 각 image 를 분류  
- 마지막 convolutional layer 에서 출력되는 feature 이용
- image 를 InceptionV3 input size 로 resize :  299px by 299px

* [preprocess_input](https://www.tensorflow.org/api_docs/python/tf/keras/applications/inception_v3/preprocess_input) method 를 이용하여 image normalize (-1 to 1 사이로 pixel 값 조정)

In [0]:
def load_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (299, 299))
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return img, image_path

## pretrained Imagenet weights 로 InceptionV3 초기화

- 마지막 convolutional layer 의 shape 은 ```8x8x2048``` 이고 이 resulting vector 를 이용하여 attention 작성

- 각 image 의 resulting vector 를 dictoinary(image_name --> feature_vector) 에 저장  
- dictionary 를 pickle file 로 disk 에 저장

In [0]:
image_model = tf.keras.applications.inception_v3.InceptionV3(weights='imagenet')
new_input = image_model.input
hidden_layer = image_model.layers[-3].output

image_features_extract_model = tf.keras.Model(new_input, hidden_layer)

In [0]:
image_features_extract_model.layers[-1].output

## InceptionV3 에서 출력된 feature 를 cache 에 저장

- InceptionV3 로 image pre-processing 및 output 을 disk 에 caching  (image 당 8 \* 8 \* 2048 floats)

- caching 은 Colab GPU 작업 시 약 10 분.  


In [0]:
save_folder = "/content/drive/My Drive/my_Flicker8/"
if not os.path.exists(save_folder):
  os.mkdir(save_folder)

In [0]:
s = time.time()
cnt = 0

# Get unique images
BATCH_SIZE = 16
encode_train = sorted(set(img_name_vector))

image_dataset = tf.data.Dataset.from_tensor_slices(encode_train)
image_dataset = image_dataset.map(load_image).batch(BATCH_SIZE)

for img, path in tqdm(image_dataset):
    batch_features = image_features_extract_model(img)
    batch_features = tf.reshape(batch_features,
                              (batch_features.shape[0], -1, batch_features.shape[3]))
    print(batch_features.shape)
    for bf, p in zip(batch_features, path):
        path_of_feature = p.numpy().decode("utf-8")
        file_path = path_of_feature.split('Flicker8k_Dataset/')
        path_of_feature = save_folder + file_path[1]
        np.save(path_of_feature, bf.numpy())
        
print(f"image 의 feature vector 변환 시간 : {cnt} 건, {time.time() - s}")

In [0]:
print(batch_features.shape)
print(path.shape)
print(path[0])
print(bf.shape)
path_of_feature

## Preprocess and tokenize the captions

- caption 을 tokenize  

- vocaburary size 를 5000 으로 제한. 다른 단어들은 "UNK" 로 치환  

- word-to-index 와 index-to-word mapping 생성  

- 모든 sequence 가 가장 긴 것을 기준으로 같은 길이가 되도록 zero padding

In [0]:
# Find the maximum length of any caption in our dataset
def calc_max_length(tensor):
    return max(len(t) for t in tensor)

In [0]:
# Choose the top 5000 words from the vocabulary
top_k = 5000
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=top_k,
                                                  oov_token="<unk>",
                                                  filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ ')
tokenizer.fit_on_texts(train_captions)
train_seqs = tokenizer.texts_to_sequences(train_captions)

In [0]:
tokenizer.word_index['<pad>'] = 0
tokenizer.index_word[0] = '<pad>'

In [0]:
# tokenized vector 생성
train_seqs = tokenizer.texts_to_sequences(train_captions)

In [0]:
# 각 vector 들을 caption 의 max_length 로 padding
cap_vector = tf.keras.preprocessing.sequence.pad_sequences(train_seqs, padding='post')

In [0]:
# attention weight 를 저장하는데 사용할 max_length 계산
max_length = calc_max_length(train_seqs)

## Split the data into training and testing

In [0]:
# Create training and validation sets using an 80-20 split
img_name_train, img_name_val, cap_train, cap_val = train_test_split(img_name_vector,
                                                                    cap_vector,
                                                                    test_size=0.2,
                                                                    random_state=0)

In [0]:
len(img_name_train), len(cap_train), len(img_name_val), len(cap_val)

## Create a tf.data dataset for training



In [0]:
BATCH_SIZE = 64
BUFFER_SIZE = 1000
embedding_dim = 256
units = 512
vocab_size = top_k + 1
num_steps = len(img_name_train) // BATCH_SIZE

# InceptionV3 의 feature shape 이 (64, 2048) 이므로
features_shape = 2048
attention_features_shape = 64

In [0]:
# 미리 저장해 놓았던 numpy feature file load
def map_func(img_name, cap):
    img_path = img_name.decode('utf-8')
    file_path = img_path.split('Flicker8k_Dataset/')
    path_of_feature = save_folder + file_path[1]
    img_tensor = np.load(path_of_feature +'.npy')
    return img_tensor, cap

In [0]:
dataset = tf.data.Dataset.from_tensor_slices((img_name_train, cap_train))

# map 함수를 사용하여 numpy file load 를 병행처리 하도록 함
dataset = dataset.map(lambda item1, item2: tf.numpy_function(
          map_func, [item1, item2], [tf.float32, tf.int32]),
          num_parallel_calls=tf.data.experimental.AUTOTUNE)       # 병행처리

# Shuffle and batch 생성
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
dataset = dataset.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

## Model

- coder 부분은  [Neural Machine Translation with Attention](../sequences/nmt_with_attention.ipynb) 과 동일  

- visual encoder 가 기계번역의 decoder 와 차이나는 부분  
* InceptionV3 의 마지막 convolutional layer 에서 (8, 8, 2048) vector 를 얻음  
* 이 것을 (64, 2048) 로 reshape
* 이 vector 가 단일 Dense layer 로 구성된 CNN Encoder 로 보내짐 
* RNN (LSTM or GRU) 가 이 feature 를 이용하여  next word 예측

In [0]:
class BahdanauAttention(tf.keras.Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)

    def call(self, features, hidden):
        # features(CNN_encoder output) shape == (batch_size, 64, embedding_dim)

        # hidden shape == (batch_size, hidden_size)
        # hidden_with_time_axis shape == (batch_size, 1, hidden_size)
        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        # score shape == (batch_size, 64, hidden_size)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))

        # attention_weights shape == (batch_size, 64, 1)
        # you get 1 at the last axis because you are applying score to self.V
        attention_weights = tf.nn.softmax(self.V(score), axis=1)

        # context_vector shape after sum == (batch_size, hidden_size)
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

In [0]:
class CNN_Encoder(tf.keras.Model):
    # Since you have already extracted the features and dumped it using pickle
    # This encoder passes those features through a Fully connected layer
    def __init__(self, embedding_dim):
        super(CNN_Encoder, self).__init__()
        # shape after fc == (batch_size, 64, embedding_dim)
        self.fc = tf.keras.layers.Dense(embedding_dim)

    def call(self, x):
        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

In [0]:
class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super(RNN_Decoder, self).__init__()
        self.units = units

        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(self.units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')
        self.fc1 = tf.keras.layers.Dense(self.units)
        self.fc2 = tf.keras.layers.Dense(vocab_size)

        self.attention = BahdanauAttention(self.units)

    def call(self, x, features, hidden):
        # defining attention as a separate model
        context_vector, attention_weights = self.attention(features, hidden)

        # x shape after passing through embedding == (batch_size, 1, embedding_dim)
        x = self.embedding(x)

        # x shape after concatenation == (batch_size, 1, embedding_dim + hidden_size)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        # passing the concatenated vector to the GRU
        output, state = self.gru(x)

        # shape == (batch_size, max_length, hidden_size)
        x = self.fc1(output)

        # x shape == (batch_size * max_length, hidden_size)
        x = tf.reshape(x, (-1, x.shape[2]))

        # output shape == (batch_size * max_length, vocab)
        x = self.fc2(x)

        return x, state, attention_weights

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

In [0]:
encoder = CNN_Encoder(embedding_dim)
decoder = RNN_Decoder(embedding_dim, units, vocab_size)

- SparseCategoricalCrossentropy  
    If reduction is NONE, this has shape [batch_size, d0, .. dN-1]; otherwise, it is scalar. (Note dN-1 because all loss functions reduce by 1 dimension, usually axis=-1.)

In [0]:
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
                    from_logits=True, reduction='none')

def loss_function(real, pred):     

    # batch 64 개 record 중 timestep t 에 0 padding 이 아닌 
    # 실제 단어가 존재하는 record 만 True 로 만듦
    mask = tf.math.logical_not(tf.math.equal(real, 0))    

    # [word_index] 에 대한 확률분포 array - (64, ), dtype=float32 
    loss_ = loss_object(real, pred)    

    # mask dtype 을 float32 로 type cast
    mask = tf.cast(mask, dtype=loss_.dtype)   

    loss_ *= mask             # 실제 단어가 존재하는 위치 외에는 모두 0 으로 만든다

    return tf.reduce_mean(loss_) 

## Checkpoint

In [0]:
checkpoint_path = "./checkpoints/train"
ckpt = tf.train.Checkpoint(encoder=encoder,
                           decoder=decoder,
                           optimizer = optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)

In [0]:
start_epoch = 0
if ckpt_manager.latest_checkpoint:
    start_epoch = int(ckpt_manager.latest_checkpoint.split('-')[-1])
    # restoring the latest checkpoint in checkpoint_path
    ckpt.restore(ckpt_manager.latest_checkpoint)

## Training

* 해당되는 `.npy` files 에 저장했던 features 를 encoder 에 통과.
* encoder 의 output, hidden state(initialized to 0) 및 decoder input (start token) 을 decoder 에 전달  
* decoder 는 prediction 과 decoder 의 hidden state 를 반환  
* decoder 의 hidden state 가 model 로 되돌려지고, prediction 은 loss 계산에 사용.  
* teacher forcing 을 사용하여 decoder 의 next input 결정 
* gradient 를 계산하여 optimizer 와 backpropagate 에 적용.


In [0]:
loss_plot = []

In [0]:
@tf.function
def train_step(img_tensor, target):
    loss = 0

    # initializing the hidden state for each batch
    # because the captions are not related from image to image
    hidden = decoder.reset_state(batch_size=target.shape[0])

    dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * target.shape[0], 1)

    with tf.GradientTape() as tape:
        
        features = encoder(img_tensor)

        for i in range(1, target.shape[1]):
            # passing the features through the decoder
            predictions, hidden, _ = decoder(dec_input, features, hidden)

            loss += loss_function(target[:, i], predictions)

            # using teacher forcing
            dec_input = tf.expand_dims(target[:, i], 1)

    total_loss = (loss / int(target.shape[1]))

    trainable_variables = encoder.trainable_variables + decoder.trainable_variables

    gradients = tape.gradient(loss, trainable_variables)

    optimizer.apply_gradients(zip(gradients, trainable_variables))

    return loss, total_loss

In [0]:
EPOCHS = 10

for epoch in range(start_epoch, EPOCHS):
    start = time.time()
    total_loss = 0

    for (batch, (img_tensor, target)) in enumerate(dataset):
        batch_loss, t_loss = train_step(img_tensor, target)
        total_loss += t_loss

        if batch % 100 == 0:
            print ('Epoch {} Batch {} Loss {:.4f}'.format(
              epoch + 1, batch, batch_loss.numpy() / int(target.shape[1])))
    # storing the epoch end loss value to plot later
    loss_plot.append(total_loss / num_steps)

    if epoch % 5 == 0:
        ckpt_manager.save()

    print ('Epoch {} Loss {:.6f}'.format(epoch + 1,
                                         total_loss/num_steps))
    print ('Time taken for 1 epoch {} sec\n'.format(time.time() - start))

In [0]:
plt.plot(loss_plot)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Loss Plot')
plt.show()

## Caption!

* evaluate 함수는 training loop 과 비슷하지만 teacher forcing 을 하지 않고 각 time step 별 decoder input 은 이전 step 의 prediction 이 된다. 
* end token 이 나오면 prediction 을 중지  
* 매 time step 별로 attention weight 를 저장

In [0]:
def evaluate(image):
    attention_plot = np.zeros((max_length, attention_features_shape))

    hidden = decoder.reset_state(batch_size=1)

    temp_input = tf.expand_dims(load_image(image)[0], 0)
    img_tensor_val = image_features_extract_model(temp_input)
    img_tensor_val = tf.reshape(img_tensor_val, (img_tensor_val.shape[0], -1, img_tensor_val.shape[3]))

    features = encoder(img_tensor_val)

    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)
    result = []

    for i in range(max_length):
        predictions, hidden, attention_weights = decoder(dec_input, features, hidden)

        attention_plot[i] = tf.reshape(attention_weights, (-1, )).numpy()

        predicted_id = tf.random.categorical(predictions, 1)[0][0].numpy()
        result.append(tokenizer.index_word[predicted_id])

        if tokenizer.index_word[predicted_id] == '<end>':
            return result, attention_plot

        dec_input = tf.expand_dims([predicted_id], 0)

    attention_plot = attention_plot[:len(result), :]
    return result, attention_plot

In [0]:
def plot_attention(image, result, attention_plot):
    temp_image = np.array(Image.open(image))

    fig = plt.figure(figsize=(5, 5))
    plt.imshow(temp_image)

    fig = plt.figure(figsize=(10, 10))
    len_result = len(result)
    for l in range(len_result):
        temp_att = np.resize(attention_plot[l], (8, 8))
        ax = fig.add_subplot(len_result//2, len_result//2, l+1)
        ax.set_title(result[l])
        img = ax.imshow(temp_image)
        ax.imshow(temp_att, cmap='gray', alpha=0.6, extent=img.get_extent())

    plt.tight_layout()
    plt.show()

In [0]:
# captions on the validation set
rid = np.random.randint(0, len(img_name_val))
image = img_name_val[rid]
real_caption = ' '.join([tokenizer.index_word[i] for i in cap_val[rid] if i not in [0]])
result, attention_plot = evaluate(image)

print ('Real Caption:', real_caption)
print ('Prediction Caption:', ' '.join(result))
plot_attention(image, result, attention_plot)


## Try it on your own images

In [0]:
image_url = 'https://tensorflow.org/images/surf.jpg'
image_extension = image_url[-4:]
image_path = tf.keras.utils.get_file('image'+image_extension,
                                     origin=image_url)

result, attention_plot = evaluate(image_path)
print ('Prediction Caption:', ' '.join(result))
plot_attention(image_path, result, attention_plot)